<a href="https://colab.research.google.com/github/programmermahi/AL_Assignment/blob/main/decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import numpy as np

In [14]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.cluster import DBSCAN
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix

In [15]:
# Load dataset
df = pd.read_csv('/content/ecommerce_customers.csv')
df.head()

,CustomerID,Age,Gender,AnnualSpending,PurchaseFrequency,TimeOnWebsite,MembershipType,PreferredCategory,LoyaltyLevel
0,C0001,65,Female,98810,9,42,Silver,Fashion,Medium
1,C0002,23,Female,33289,8,309,Silver,Electronics,Low
2,C0003,50,Male,27280,15,149,Platinum,Fashion,Low
3,C0004,44,Male,108850,43,309,Gold,Beauty,High
4,C0005,62,Female,51463,2,111,Gold,Groceries,Medium


In [16]:
label_encoder = LabelEncoder()
df['LoyaltyLevel_encoded'] = label_encoder.fit_transform(df['LoyaltyLevel'])
print("LoyaltyLevel encoded successfully.")
print(df[['LoyaltyLevel', 'LoyaltyLevel_encoded']].head())

LoyaltyLevel encoded successfully.
  LoyaltyLevel  LoyaltyLevel_encoded
0       Medium                     2
1          Low                     1
2          Low                     1
3         High                     0
4       Medium                     2


In [17]:
categorical_features = ['Gender', 'MembershipType', 'PreferredCategory']
for feature in categorical_features:
    df[f'{feature}_encoded'] = label_encoder.fit_transform(df[feature])
print("Categorical features encoded successfully.")
print(df[['Gender', 'Gender_encoded', 'MembershipType', 'MembershipType_encoded', 'PreferredCategory', 'PreferredCategory_encoded']].head())

Categorical features encoded successfully.
   Gender  Gender_encoded MembershipType  MembershipType_encoded  \
0  Female               0         Silver                       2   
1  Female               0         Silver                       2   
2    Male               1       Platinum                       1   
3    Male               1           Gold                       0   
4  Female               0           Gold                       0   

  PreferredCategory  PreferredCategory_encoded  
0           Fashion                          3  
1       Electronics                          2  
2           Fashion                          3  
3            Beauty                          0  
4         Groceries                          4  


In [18]:
scaler = StandardScaler()
numeric_features = ['Age', 'AnnualSpending', 'PurchaseFrequency', 'TimeOnWebsite']
for feature in numeric_features:
    df[f'{feature}_scaled'] = scaler.fit_transform(df[[feature]])

print("Numeric features normalized successfully.")
print(df[numeric_features + [f'{f}_scaled' for f in numeric_features]].head())

Numeric features normalized successfully.
   Age  AnnualSpending  PurchaseFrequency  TimeOnWebsite  Age_scaled  \
0   65           98810                  9             42    1.670842   
1   23           33289                  8            309   -1.286027   
2   50           27280                 15            149    0.614818   
3   44          108850                 43            309    0.192408   
4   62           51463                  2            111    1.459637   

   AnnualSpending_scaled  PurchaseFrequency_scaled  TimeOnWebsite_scaled  
0               1.007912                 -1.149389             -1.603177  
1              -1.184605                 -1.227019              1.251544  
2              -1.385683                 -0.683609             -0.459150  
3               1.343879                  1.490029              1.251544  
4              -0.576452                 -1.692798             -0.865440  


In [19]:
features = [
    'Age_scaled',
    'AnnualSpending_scaled',
    'PurchaseFrequency_scaled',
    'TimeOnWebsite_scaled',
    'Gender_encoded',
    'MembershipType_encoded',
    'PreferredCategory_encoded'
]
X = df[features]
y = df['LoyaltyLevel_encoded']

print("Features DataFrame (X) head:")
print(X.head())
print("\nTarget Series (y) head:")
print(y.head())

Features DataFrame (X) head:
   Age_scaled  AnnualSpending_scaled  PurchaseFrequency_scaled  \
0    1.670842               1.007912                 -1.149389   
1   -1.286027              -1.184605                 -1.227019   
2    0.614818              -1.385683                 -0.683609   
3    0.192408               1.343879                  1.490029   
4    1.459637              -0.576452                 -1.692798   

   TimeOnWebsite_scaled  Gender_encoded  MembershipType_encoded  \
0             -1.603177               0                       2   
1              1.251544               0                       2   
2             -0.459150               1                       1   
3              1.251544               1                       0   
4             -0.865440               0                       0   

   PreferredCategory_encoded  
0                          3  
1                          2  
2                          3  
3                          0  
4               

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [25]:
dt_classifier = DecisionTreeClassifier(random_state=42, criterion='gini')
dt_classifier.fit(X_train, y_train)


DecisionTreeClassifier(random_state=42)

In [26]:
y_pred = dt_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)

Model Accuracy: 1.0000

Confusion Matrix:
[[ 35   0   0]
 [  0  45   0]
 [  0   0 120]]


In [23]:
feature_importances = dt_classifier.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("Feature Importances (sorted descending):")
print(importance_df)

Feature Importances (sorted descending):
                     Feature  Importance
1      AnnualSpending_scaled     0.81764
2   PurchaseFrequency_scaled     0.18236
0                 Age_scaled     0.00000
3       TimeOnWebsite_scaled     0.00000
4             Gender_encoded     0.00000
5     MembershipType_encoded     0.00000
6  PreferredCategory_encoded     0.00000
